In [1]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import namedtuple, deque
import torch.optim as optim
import datetime
import gymnasium as gym
import matplotlib.pyplot as plt
from scipy.special import softmax
import numpy as np
from collections import deque, namedtuple
from torch.distributions import Categorical

In [2]:
env = gym.make('CartPole-v1')
env.reset(seed=0)

(array([ 0.01369617, -0.02302133, -0.04590265, -0.04834723], dtype=float32),
 {})

In [3]:
class QNetwork1(nn.Module):

    def __init__(self, state_size, action_size, seed,adv_type = 'avg', fc1_units=128, fc2_units=64,fc3_units = 256):
        super(QNetwork1, self).__init__()

        self.fc1 = nn.Linear(state_size,fc1_units)
        self.fc2 = nn.Linear(fc1_units,fc2_units)
        self.fc3 = nn.Linear(fc2_units,fc3_units)

        self.value = nn.Linear(fc3_units,1)

        self.adv = nn.Linear(fc3_units, action_size)

    def forward(self,x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))

        adv = self.adv(x)

        value = self.value(x)

        #just go with mean method
        # set dim to 1, as we have batch input and dont want mess up
        adv_mean = torch.mean(adv,dim=1,keepdim=True)
        q = value + adv - adv_mean
        return q
        
        
  

In [4]:
BUFFER_SIZE = int(1e5)  # replay buffer size
BATCH_SIZE = 64         # minibatch size
GAMMA = 0.99            # discount factor
LR = 5e-4               # learning rate
UPDATE_EVERY = 20       # how often to update the network (When Q target is present)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

class Buffer:

    def __init__(self,action_size, buffer_size, batch_size, seed):

        self.memory = deque(maxlen=BUFFER_SIZE)
        self.batch_size = batch_size
        self.seed = random.seed(seed)

    def add(self, state, action, reward, next_state,done):

        self.memory.append([state,action,reward,next_state,done])

    def sample(self):

        experience = random.sample(self.memory,k=self.batch_size)

        state_bch,action_bch,reward_bch,next_state_bch,done_bch = zip(*experience)

        state_bch = torch.FloatTensor(state_batch).to(device) 
        action_bch = torch.FloatTensor(action_bch).to(device) 
        reward_bch = torch.FloatTensor(reward_bch).to(device) 
        next_state_bch = torch.FloatTensor(next_state_bch).to(device) 
        done_bch = torch.FloatTensor(done_bch).to(device) 

        return state_bch, action_bch, reward_bch, next_state_bch, done_bch
        

    def __len__(self):

        return len(self.memory())

cuda:0
